# T08. Everything is an object

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/t08-everything-is-an-object/t08.ipynb)

T07 ended with the interpreter loop pushing and popping things. This lesson is about the things.

![the eight stages of the pipeline with none of them highlighted](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/where-we-are.svg)

Nothing is lit up. Earlier lessons each owned a box; this one is about what travels along the arrows: the same kind of value at every stage, a `PyObject`.

The sentence "everything in Python is an object" is more literal than it sounds: an integer is an [object](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#object), so is a string, a function, the type of that function, and the [frame](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#frame) you are running in. They all begin with the same two fields, which is how one loop pushes any of them around without knowing what they are.

By the end you will be able to answer four questions about any value on your screen, know why `257 is 257` says True for a reason nobody expects, and why `sys.getrefcount` changed its answer in 3.14.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/object.h:127-150@v3.15.0rc1#_object`.

Read it as four parts: the file, the lines, the release those line numbers belong to, and the name of the thing they are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Two of the numbers in this lesson changed in 3.15. Everything here was checked against the version this cell prints, and against 3.14, and it says out loud where the two disagree.

In [ ]:
import pyxray

pyxray.show()

## Predict first

Before anything else, four one line questions. Write your four answers down somewhere, then run the cell.

Does `256 is 256` say True? Does `257 is 257`? Does `"a" * 1 is "a"`? Does `[] is []`?

Guessing wrong here is the useful part. Three of these four have an answer that is right for a reason almost nobody gets right, and the rest of the lesson is about that gap.

In [ ]:
one = 256
two = 256
print("256 is 256      ->", one is two)

three = 257
four = 257
print("257 is 257      ->", three is four)

letter = "a"
built = "a" * 1
print("'a' * 1 is 'a'  ->", built is letter)

first = []
second = []
print("[] is []        ->", first is second)

True, True, True, False.

If you predicted False for the second because you read that the cache stops at 256, you had the right idea and the wrong experiment. If you predicted True because you read that 3.15 raised the limit, you got the right answer for the wrong reason. We come back to this once there is enough on the table to explain it.

## What every object starts with

Every value in a running Python has the same thing sitting in front of it.

![the object header as three stacked fields: refcount, type pointer, then the type specific data](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/the-header.svg)

every object in a running Python starts with the same two fields, a reference count and a pointer to its type, and then whatever the specific type needs. Those two fields are the [object header](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#object-header), and the C for it is [Include/object.h:127-150@v3.15.0rc1#_object](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L127-L150), shorter than most people expect for the most important [struct](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#struct) in the codebase.

That shape is what makes the rest work. The [eval loop](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#eval-loop) from T07 pushes and pops `PyObject *`, a [pointer](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#pointer) to those two fields, so it never needs to know whether the thing on the end is a dictionary or a socket. For behaviour it follows `ob_type` and asks the type, and when it is done with a value it decrements `ob_refcnt`.

The free threaded build has a wider header, because a plain increment is not safe when two threads do it at once: [Include/object.h:156-170@v3.15.0rc1#_object](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L156-L170). Same idea, more fields. Everything below is about the ordinary build.

## Four questions

There are exactly four things Python will tell you about the header, and each one is easy to read too much into.

![a table of id, type, getrefcount and getsizeof with what each one does not tell you](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/four-questions.svg)

The last column is the one worth memorising, because every popular confusion about identity, memory and lifetime lives in it.

an integer, a string, a dictionary and a plain function all answer the same four questions, because all four are objects. `pyxray.obj.header` asks all four at once and prints them as a sentence.

In [ ]:
from pyxray import obj

for value in [None, 42, "hello", [1, 2, 3], {"a": 1}, obj.header]:
    print(obj.header(value).describe())

Look at the last line. `obj.header` is a function, and a function has an address, a type, a [reference count](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#reference-count) and a size just like a list does. That is the sentence "everything is an object" turned into something you can print.

Two things in that output are worth pausing on. `None` reports its reference count as parked rather than as a number, which we get to below. And the ints and strings say they are not tracked by the [cycle collector](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#cycle-collector): an int cannot hold a reference to anything, so it can never be part of a cycle, and the collector does not carry the extra header for it. Tracking is opt in per type, and T09 is where that matters.

## Two questions that look alike

`==` and `is` get taught next to each other and they are not related.

![== calls the type, is compares two addresses, side by side](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/is-versus-equals.svg)

`==` is a method call. `int.__eq__` compares values, `str.__eq__` compares characters, and your own class can make it mean whatever you like.

`is` compares two addresses. The [instruction](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#instruction) is [Python/bytecodes.c:3363-3370@v3.15.0rc1#_IS_OP](https://github.com/python/cpython/blob/v3.15.0rc1/Python/bytecodes.c#L3363-L3370), and the whole implementation is one call to `Py_Is`, which is a pointer comparison. No type is consulted and no method is called, which is why you cannot override it and why it is fast.

So `a is b` asks whether two names point at the same object, and `a == b` asks whether two objects agree that they are equal. Different questions, and two lists built from the same three numbers are equal and are not the same object.

In [ ]:
a = [1, 2, 3]
b = [1, 2, 3]
c = a

print("a == b", a == b, "  same contents")
print("a is b", a is b, "  different objects")
print("a is c", a is c, "  same object")
print()
print("id(a)", hex(id(a)))
print("id(b)", hex(id(b)))
print("id(c)", hex(id(c)))

`c = a` did not copy anything. It bound a second name to the same object, which is why the two ids match, and why appending to `a` changes what `c` sees.

The rule is short. Use `is` for `None`, `True`, `False` and sentinels you made yourself, where you genuinely mean this exact object. Use `==` for everything else.

## The shelf

Small integers get special treatment, and it is worth knowing exactly what the treatment is.

![the small integer cache drawn as a row of prebuilt boxes with everything past the end built fresh](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/the-shelf.svg)

CPython builds a block of integer objects while it is starting up, before your code runs, and hands out pointers into that block whenever an arithmetic result lands in range. That block is the [small integer cache](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#small-integer-cache), and the handing out is one line: [Objects/longobject.c:60-65@v3.15.0rc1#get_small_int](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/longobject.c#L60-L65), which indexes an array rather than allocating anything.

The size of the array is a number in a [header file](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#header-file): [Include/internal/pycore_runtime_structs.h:96-98@v3.15.0rc1#_PY_NSMALLPOSINTS](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_runtime_structs.h#L96-L98). That is where the famous 256 came from, and the top of the small integer cache moved in 3.15, so the 256 that every tutorial quotes is the old number.

It exists because small integers are everywhere: loop counters, lengths, indexes, flags. Allocating a fresh object for every `i + 1` would be a lot of allocation for a handful of distinct values.

Rather than trusting either number, ask your own interpreter: where the sharing stops can be measured from Python, by walking outward from zero until two equal integers stop being the same object. That is all `pyxray.obj.small_int_range` does.

In [ ]:
from pyxray import obj

low, high = obj.small_int_range()
print(f"this interpreter shares integers from {low} to {high}")

for value in [-6, -5, 0, 255, 256, 257, 1024, 1025]:
    shared = "shared" if obj.shares_identity(value) else "built fresh"
    print(f"   {value:>6}   {shared}")

Note how the probe builds its integers: `int(str(n))` rather than the literal twice, and the next section is about why that matters.

## Two reasons to say True

The famous `257 is 257` example gets one thing wrong, and it is worth being exact about which thing.

![a = 257 answered by the compiler, a = int('257') answered by the cache](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/two-reasons-to-say-true.svg)

When you write `a = 257` and `b = 257` in the same cell, both lines compile into one code object, and the compiler stores each distinct constant once. Both `LOAD_CONST` instructions then point at the same object, so two identical integer literals in one piece of source become one object however big the number is, which is the compiler keeping each distinct constant once and has nothing to do with the small integer cache.

You can see it directly in `co_consts`.

In [ ]:
source = "a = 257\nb = 257\n"
code = compile(source, "<demo>", "exec")

print("constants the compiler kept:", code.co_consts)

scope = {}
exec(code, scope)
print("a is b ->", scope["a"] is scope["b"], "  because there is one constant, not two")

> **Version note.** On 3.14 the implicit return None at the end is a LOAD_CONST and None sits in co_consts, so you get one more constant and two fewer bytes of bytecode than the text says.

One 257 in the tuple and two instructions loading it, so the cache never came into it.

Now the same question asked in a way the compiler cannot answer ahead of time. building the same integer twice through int() rather than writing it as a literal does show the cache, and the sharing stops once the value is past the top of it.

In [ ]:
def fresh(text):
    return int(text)


print("int('257') twice  ->", fresh("257") is fresh("257"))
print("int('10') twice   ->", fresh("10") is fresh("10"))
print("int('99999') twice->", fresh("99999") is fresh("99999"))

That last block is measuring the shelf. On 3.15 the first two say True and the third says False. On 3.14 the first says False, because 257 is past the old limit.

So `257 is 257` says True on both versions for a compiler reason, and it would still say True if the cache were deleted tomorrow. Every tutorial that uses it to demonstrate the cache is demonstrating the compiler instead. The docs have always said the same thing: identity of immutable values is not something the language promises, and it has already changed once inside this project's lifetime.

## Strings get the same treatment, with a rule

Strings have their own version of the shelf, the intern table, and putting a string into it is called [interning](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#interning). It is a hash table hanging off the interpreter state, at [Include/internal/pycore_runtime_structs.h:91-94@v3.15.0rc1#_Py_cached_objects](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_runtime_structs.h#L91-L94), and a string in it is shared by everything that asks for an equal string. Not every string goes in, and the rule is about what the string looks like.

![six string literals with whether they are interned and why](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/what-gets-interned.svg)

The check is fifteen lines and it is what you would guess if you knew what interning was for: [Objects/codeobject.c:116-137@v3.15.0rc1#should_intern_string](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/codeobject.c#L116-L137). The string has to be ASCII with every character alphanumeric or an underscore, which is another way of saying strings shaped like identifiers.

That is what the table is for. Attribute lookups, variable names, keyword arguments and dictionary keys get compared constantly while your program runs, and comparing two pointers is faster than comparing two sets of characters. A sentence with a space in it is not worth the table entry, so a string is interned only when it is ASCII and shaped like an identifier, which puts append and _private in the table and leaves hello world out of it.

`pyxray.obj.is_interned` asks the question without changing the answer, which takes care: interning the string you were handed would put it in the table, and every call after the first would say True.

In [ ]:
from pyxray import obj

for text in ["", "a", "append", "_private", "x1", "hello world", "a-b"]:
    print(f"{text!r:<15} interned: {obj.is_interned(text)}")

That is the same list as the table above, measured on your interpreter rather than quoted from the prose.

Two things worth knowing, and the cell below shows both. a string built at runtime is not interned even when it is shaped like an identifier, and sys.intern puts it in the table and hands back the object that is in there. The first half is because interning happens when a code object is created, and a string built by `join` was never a constant in one. The second half is worth doing if you are about to compare the same string a few million times and worth ignoring otherwise.

In [ ]:
import sys

from pyxray import obj

built = "".join(["app", "end"])
print("built at runtime, interned:", obj.is_interned(built))

kept = sys.intern(built)
print("after sys.intern, interned:", obj.is_interned(built))
print("and it handed back the same object:", kept is built)

## Counting who is holding it

The reference count is the first field in the header, and it is the number that decides when an object goes away. Every place holding the object adds one, and when it hits zero the object is freed on the spot.

Asking Python for that number turns out to be harder than it sounds.

![LOAD_FAST_BORROW adds nothing, LOAD_GLOBAL adds one, side by side](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/borrowed-or-not.svg)

`sys.getrefcount` is documented as reporting one more than you expect, because passing the object to the function creates a reference. Every tutorial written before 3.14 says to subtract one and move on.

In 3.14 that stopped being reliable. `LOAD_FAST_BORROW` arrived, and loading a local now hands over a [borrowed reference](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#borrowed-reference) rather than a counted one, because the frame is already holding the object. Loading a global still takes a real reference, since nothing guarantees the global survives.

So the correction depends on the instruction that pushed the argument, and sys.getrefcount gives a different answer for a local and a global that each hold one list, because loading a local borrows the reference and loading a global takes one.

In [ ]:
import sys

from pyxray import obj

GLOBAL_LIST = [1, 2, 3]


def compare():
    local_list = [1, 2, 3]
    print("            sys.getrefcount   pyxray")
    print(f"local       {sys.getrefcount(local_list):>13}   {obj.refcount(local_list):>6}")
    print(f"global      {sys.getrefcount(GLOBAL_LIST):>13}   {obj.refcount(GLOBAL_LIST):>6}")


compare()

Both lists are held in exactly one place. The raw numbers disagree and the corrected ones do not.

`pyxray.obj.refcount` gets there by disassembling the caller, looking at the instruction immediately before the call, and subtracting one only if that instruction took a real reference. That is a lot of work for one number, and the alternative is showing a beginner 0 references for a variable they just bound.

Here is the count moving as containers pick the object up and put it down again. every container holding an object holds a reference of its own, so putting one list inside another list twice raises its count by two and clearing that list drops it back.

In [ ]:
from pyxray import obj


def show(label, value):
    print(f"{label:<28} {obj.refcount(value)}")


def watch():
    thing = [1, 2, 3]
    show("just bound", thing)

    holder = [thing, thing]
    show("also in a list twice", thing)

    box = {"key": thing}
    show("also in a dict", thing)

    holder.clear()
    show("list cleared", thing)

    del box
    show("dict gone", thing)


watch()

One, three, four, two, one. Each container that holds the object holds a reference, and dropping the container drops the reference.

The whole thing is wrapped in a function on purpose. At the top level of a notebook a name lives in the module dictionary, which holds a reference too, so every number above would be one higher and the first would read 2 for a thing you just made. The count is real, and working out which places it is counting is on you.

You can also ask the other direction: what is holding this thing right now. Python can be asked which objects are holding a value, and at the top level of a notebook one of the answers is always the module's own namespace.

In [ ]:
from pyxray import obj

target = ["watch me"]
somewhere = {"target": target}
elsewhere = [target]

for holder in obj.referrers(target):
    print(holder)

Two dicts and a list. The list and one dict are the containers the cell built. The other dict is the notebook's own namespace, holding `target` because `target` is a name at the top level, which is the extra reference the previous cell went out of its way to avoid.

`gc.get_referrers` does the work, and `pyxray` filters out the frames: the frame you are asking from is holding the object precisely because you are asking, which is an artifact rather than an answer.

## The objects that are never freed

Some objects have their reference count parked at a value the interpreter never decrements. `None`, `True`, `False`, the small integers, the interned strings and every [type object](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#type-object) are [immortal](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#immortal-object). The check is a sign test: [Include/refcount.h:125-136@v3.15.0rc1#_Py_IsImmortal](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L125-L136).

The reason is threading. Incrementing a shared counter means writing to a cache line, so every core touching `None` a million times a second means those cores fighting over one line. Freezing the count for objects that will never be freed makes the write unnecessary. This landed in 3.12 and it is what made the free threaded build plausible.

It also means None, True, the small integers and the type objects have their reference count parked, so sys.getrefcount reports an enormous number for them that is not counting anything, which is why `pyxray` reports nothing for those.

In [ ]:
import sys

from pyxray import obj


def report(label, value):
    if obj.is_immortal(value):
        print(f"{label:<15} immortal, raw count reads {sys.getrefcount(value)}")
    else:
        print(f"{label:<15} ordinary, {obj.refcount(value)} reference(s)")


report("None", None)
report("True", True)
report("the int 5", 5)
report("str", str)
report("a fresh list", [])

## How big is it

The last of the four questions. `sys.getsizeof` asks the object itself, by calling its `__sizeof__` method, and then adds the garbage collector's pre header if the type has one: [Python/sysmodule.c:1931-1946@v3.15.0rc1#_PySys_GetSizeOf](https://github.com/python/cpython/blob/v3.15.0rc1/Python/sysmodule.c#L1931-L1946).

Start with things that are holding nothing at all.

![a bar chart of the size of None, 42, a one character string, an empty tuple, list and dict](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/sizes.svg)

Those are the header plus whatever the type needs in order to exist. An int carries a sign, a length and at least one digit. A one character string carries its length, its hash, a flag saying it is ASCII, and the character. A list carries a length, a pointer to its array of slots, and a capacity.

Now watch a list fill up.

![a bar chart of an empty list against lists of ten, a hundred and a thousand items](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/t08-everything-is-an-object/diagrams/sizes-grow.svg)

a list costs one pointer per slot, so what an extra item adds is the size of an address on your machine and not the size of the thing you put in.

How big a word is depends on the machine, so the next cell measures it rather than telling you.

In [ ]:
import ctypes
import sys

word = ctypes.sizeof(ctypes.c_void_p)
empty = sys.getsizeof([])
ten = sys.getsizeof([0] * 10)

print("one pointer here: ", word, "bytes")
print("an empty list:    ", empty, "bytes")
print("a list of ten:    ", ten, "bytes")
print("so one slot costs:", (ten - empty) // 10, "bytes")

> **Version note.** A pointer is 8 bytes on an ordinary 64 bit machine and 4 in a browser, where Python is compiled to WebAssembly as a 32 bit build. Every number in this cell moves with it.

On a laptop or a desktop that last line says eight, because the addresses are 64 bit. In a browser it says four, because the Python running there is a 32 bit WebAssembly build, and the sizes in this whole section shrink with it. Neither number is a fact about Python: the fact about Python is one pointer per slot.

That is also the trap in the fourth question: sys.getsizeof reports an object's own bytes and nothing it points at, so a list of three tiny integers and a list of three enormous ones come out the same size.

In [ ]:
import sys

small = [1, 2, 3]
big = [10**100, 10**100, 10**100]

print("list of three small ints:", sys.getsizeof(small), "bytes")
print("list of three huge ints: ", sys.getsizeof(big), "bytes")
print()
print("one huge int on its own: ", sys.getsizeof(big[0]), "bytes")
print()
really = sys.getsizeof(big) + sum(sys.getsizeof(n) for n in big)
print("the second list really costs about", really, "bytes")

The two lists are the same size, because they are the same three pointers. Everything that makes the second expensive is on the other end of those pointers, and `getsizeof` will not follow them. Nothing in the standard library will, because "how big is this really" runs into shared objects and cycles and stops having one answer.

## Try it yourself

**One.** Find the exact integer where sharing stops on your interpreter without using `small_int_range`. Then explain why writing the literal twice in one cell gives you the wrong boundary.

**Two.** `sys.getsizeof([1])` is not `sys.getsizeof([])` plus one slot. Find out what both actually are on your machine, then append items one at a time and print the size whenever it changes. The pattern you get is the list's growth strategy, and it is in `Objects/listobject.c` if you want to check your reading.

**Three.** Build two equal strings that are not the same object, then intern both and check what `is` says. Then do it again with strings containing a space, and explain the difference.

**Four.** Take a class of your own, give it `__eq__`, and find a case where `a == b` is True and `a is b` is False. Then find out what happens to that class in a `set` and work out why `__hash__` disappeared.

**Five.** Write a function that takes any object and prints its reference count, then call it with the same list from a local variable, from a global, and from inside a list comprehension. Predict the three raw numbers before you run it.

## What just happened

Every value in a running Python starts with the same two fields: a reference count and a pointer to its type. That is what lets one interpreter loop push around integers, sockets and functions without knowing which is which.

There are four questions you can ask about an object and each has a sharp edge. `id` tells you where it is and nothing about equal objects elsewhere. `type` tells you where the behaviour lives and nothing about which instance you have. `getrefcount` tells you how many places hold it, plus however many the asking cost. `getsizeof` tells you the object's own bytes and nothing about what it points at.

`is` compares addresses and `==` calls a method. They are unrelated questions, and the two agreeing on a particular pair of values is a coincidence you should not build on.

CPython shares small integers and identifier shaped strings, because those are the values programs use constantly. Both are implementation details, both have moved, and the famous `257 is 257` example measures the compiler rather than the cache.

Reference counting is a count of holders, and in 3.14 the cost of asking stopped being a constant. Immortal objects opt out of counting entirely, which is what made the free threaded build possible.

## Where this goes next

You now know what a reference count is and when it hits zero. T09 is about what happens next: where the memory came from, why freeing it does not hand it back to the operating system, and what the cycle collector is for given that reference counting already frees things.